# Creational Patterns

## What's covered

- What the creational family solves — decoupling *who* constructs an object from *who* uses it
- **Singleton** — exactly one instance, globally accessible
- **Factory Method** — a method whose subclasses decide which concrete class to build
- **Abstract Factory** — a factory of factories, for families of related products
- **Builder** — step-by-step construction, especially for objects with many optional parts
- **Prototype** — new objects made by cloning an existing one
- How to choose between them when more than one fits


## What the creational family solves

The five creational patterns all answer the same question: **who decides which concrete class to construct?** The usual answer — "the calling code, using `new`" — is fine until it isn't. The pattern family exists for the situations where it isn't.

Three forces drive code away from direct construction:

- **The concrete type varies at runtime.** The calling code wants a `PaymentProcessor` but the choice between `Stripe`, `PayPal`, and `Square` depends on configuration, the customer's country, or A/B test bucket. The calling code should not know about the three concrete classes.
- **Construction is complex.** An object with twenty optional fields, multi-step validation, or an expensive initialization sequence is painful to build inline. A specialized builder isolates the complexity.
- **You want exactly one instance.** Or you want to clone an existing one rather than reconstructing from scratch.

In Java the answer to all three is usually "a class with `static` factory methods". In Kotlin you have `object` declarations and `data class` `.copy()` built in, which collapses two of the patterns into language features. In Python, a module is already a singleton, `dataclass` and keyword arguments cover most builders, and `copy.deepcopy()` is one line. Several of the GoF creational patterns are nearly invisible in idiomatic Python.

We still cover all five, because the intent vocabulary — "use a factory method here", "use a builder there" — is shared across the team even when the structural ceremony disappears.


## Singleton

**Intent:** ensure a class has exactly one instance, and provide a global access point to it.

**Where it shows up:** loggers, configuration objects, connection pools, registries — anything that is conceptually unique to the running process. The GoF structure has a private constructor and a static `get_instance()` method that returns the lone instance, lazily creating it on first call.

**Python's take:** a module is already a singleton. `import config; config.api_key` gives you exactly one `config` per process, no boilerplate. For when you really want a class, override `__new__` to return the cached instance.

**Kotlin's take:** the language has the pattern built in. `object Foo { ... }` declares a singleton — there is exactly one `Foo`, period. No factory method, no double-checked locking, no static state to wire up.

**The catch.** Singleton is the most over-used pattern in the catalog. Hidden global state is hard to test, hard to reason about under concurrency, and usually a signal that the caller should be receiving the dependency through its constructor instead. Reach for singleton when the object really is unique (a hardware device, the process's stdout) — not because "we only need one for now".


### Python


In [ ]:
# Approach 1 — a module is already a singleton.
# In real code this lives in its own file; here we simulate with a class.
class _Config:
    api_key: str = "default-key"
    timeout: int = 30


config = _Config()  # module-level instance; importers all see the same object


# Approach 2 — override __new__ to cache the lone instance.
class Logger:
    _instance: "Logger | None" = None

    def __new__(cls) -> "Logger":
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.buffer = []  # init only once
        return cls._instance

    def log(self, msg: str) -> None:
        self.buffer.append(msg)


a = Logger()
b = Logger()
print(a is b)         # True — same instance
a.log("first")
b.log("second")
print(a.buffer)       # ['first', 'second'] — shared state


### Kotlin

```kotlin
// The language has the pattern built in.
object Logger {
    private val buffer = mutableListOf<String>()

    fun log(msg: String) {
        buffer.add(msg)
    }

    fun all(): List<String> = buffer.toList()
}

Logger.log("first")
Logger.log("second")
println(Logger.all())   // [first, second]

// Compare to Java, where you would need: private constructor,
// private static instance, public static getInstance(),
// usually with double-checked locking for thread safety.
// Kotlin's `object` collapses all of that to one keyword.
```


**When not to use Singleton.** If you're using it because "we only need one for now" or "passing it everywhere is annoying", you're probably about to inject hidden global state into your test suite. Try **passing the dependency explicitly through constructors** first. Reach for singleton only when the object's uniqueness is part of its identity — a process-wide logger, a hardware device handle, a connection pool that exists exactly once.


## Factory Method

**Intent:** define a method for creating an object, but let subclasses decide which concrete class to instantiate.

**Where it shows up:** any framework that calls back into your code asking "what should I create here?" — Django's `get_form_class()`, JUnit's `newTestClass()`, the GoF's canonical `Dialog.create_button()` where `WindowsDialog` and `MacDialog` return different button types.

**The shape:** a base class has an algorithm (`render()`) that calls a creation method (`create_button()`) without knowing the concrete type. Subclasses override `create_button()` to control what gets built. The parent's algorithm stays unchanged; the variation lives in the subclass.

**Python's take:** a `@classmethod` factory is the idiomatic shape. Or just a plain function — Python doesn't punish you for skipping the class.

**Kotlin's take:** abstract method overridden in subclasses, or a companion object factory function.

**The contrast with Abstract Factory:** factory method is *one* product per subclass; abstract factory is a *family* of products.


### Python


In [ ]:
from abc import ABC, abstractmethod


class Button(ABC):
    @abstractmethod
    def render(self) -> str: ...


class WindowsButton(Button):
    def render(self) -> str:
        return "[Windows button]"


class MacButton(Button):
    def render(self) -> str:
        return "[Mac button]"


class Dialog(ABC):
    # the algorithm — same for every subclass
    def open(self) -> str:
        button = self.create_button()  # factory method — subclasses decide what
        return f"Dialog containing {button.render()}"

    @abstractmethod
    def create_button(self) -> Button: ...


class WindowsDialog(Dialog):
    def create_button(self) -> Button:
        return WindowsButton()


class MacDialog(Dialog):
    def create_button(self) -> Button:
        return MacButton()


print(WindowsDialog().open())  # Dialog containing [Windows button]
print(MacDialog().open())      # Dialog containing [Mac button]


# Python-idiomatic version: a classmethod factory on the product itself.
# No Dialog hierarchy needed if "which platform" is a runtime value.
class ButtonV2:
    def __init__(self, platform: str):
        self.platform = platform

    @classmethod
    def for_platform(cls, platform: str) -> "ButtonV2":
        return cls(platform)


print(ButtonV2.for_platform("mac").platform)  # mac


### Kotlin

```kotlin
sealed interface Button { fun render(): String }
class WindowsButton : Button { override fun render() = "[Windows button]" }
class MacButton : Button { override fun render() = "[Mac button]" }

abstract class Dialog {
    fun open(): String {
        val button = createButton()        // factory method
        return "Dialog containing ${button.render()}"
    }
    protected abstract fun createButton(): Button
}

class WindowsDialog : Dialog() {
    override fun createButton(): Button = WindowsButton()
}

class MacDialog : Dialog() {
    override fun createButton(): Button = MacButton()
}

println(WindowsDialog().open())
println(MacDialog().open())

// Companion object factory — the Kotlin-idiomatic alternative
// when no abstract algorithm is involved.
class ButtonV2 private constructor(val platform: String) {
    companion object {
        fun forPlatform(platform: String) = ButtonV2(platform)
    }
}

println(ButtonV2.forPlatform("mac").platform)
```


**When not to use Factory Method.** When you only ever have one concrete class — just call the constructor. The pattern earns its complexity when the parent algorithm legitimately depends on creating products whose concrete type it does not know. If "which subclass to use" is decided by a runtime value (not by polymorphism), a plain function with an `if/elif` is clearer.


## Abstract Factory

**Intent:** provide an interface for creating *families* of related products without specifying their concrete classes.

**Where it shows up:** UI toolkits (Mac vs. Windows widgets that must match), database drivers (a Postgres factory producing Postgres connections, cursors, and result sets — all matching), cloud SDKs (AWS factory vs. Azure factory producing compatible storage, queue, and identity objects).

**The shape:** an abstract `Factory` interface declares one method per product type — `create_button()`, `create_menu()`, `create_window()`. Concrete factories implement the interface to produce a consistent family. The client code holds a `Factory` reference and asks it for products without knowing which family it has.

**The distinction from Factory Method:** factory method is *one* creation method, often varied by inheritance. Abstract factory is *several* creation methods grouped, each producing a member of the same family.

**Python's take:** a class with multiple factory methods, or a dictionary of factories indexed by family name. The full ceremony with abstract base classes is rarely worth it; the intent — "products from the same family are produced together" — is what matters.


### Python


In [ ]:
from abc import ABC, abstractmethod


class Button(ABC):
    @abstractmethod
    def render(self) -> str: ...


class Menu(ABC):
    @abstractmethod
    def render(self) -> str: ...


class MacButton(Button):
    def render(self) -> str: return "[Mac button]"


class MacMenu(Menu):
    def render(self) -> str: return "[Mac menu]"


class WindowsButton(Button):
    def render(self) -> str: return "[Windows button]"


class WindowsMenu(Menu):
    def render(self) -> str: return "[Windows menu]"


class WidgetFactory(ABC):
    @abstractmethod
    def button(self) -> Button: ...

    @abstractmethod
    def menu(self) -> Menu: ...


class MacFactory(WidgetFactory):
    def button(self) -> Button: return MacButton()
    def menu(self) -> Menu:     return MacMenu()


class WindowsFactory(WidgetFactory):
    def button(self) -> Button: return WindowsButton()
    def menu(self) -> Menu:     return WindowsMenu()


def render_app(factory: WidgetFactory) -> str:
    return f"{factory.button().render()} + {factory.menu().render()}"


print(render_app(MacFactory()))      # [Mac button] + [Mac menu]
print(render_app(WindowsFactory()))  # [Windows button] + [Windows menu]


### Kotlin

```kotlin
interface Button { fun render(): String }
interface Menu   { fun render(): String }

class MacButton     : Button { override fun render() = "[Mac button]" }
class MacMenu       : Menu   { override fun render() = "[Mac menu]" }
class WindowsButton : Button { override fun render() = "[Windows button]" }
class WindowsMenu   : Menu   { override fun render() = "[Windows menu]" }

interface WidgetFactory {
    fun button(): Button
    fun menu(): Menu
}

class MacFactory : WidgetFactory {
    override fun button(): Button = MacButton()
    override fun menu(): Menu     = MacMenu()
}

class WindowsFactory : WidgetFactory {
    override fun button(): Button = WindowsButton()
    override fun menu(): Menu     = WindowsMenu()
}

fun renderApp(factory: WidgetFactory) =
    "${factory.button().render()} + ${factory.menu().render()}"

println(renderApp(MacFactory()))
println(renderApp(WindowsFactory()))
```


**When not to use Abstract Factory.** If the products are not actually a *family* — if a `MacButton` can sit happily next to a `WindowsMenu` — you don't need this pattern. Use plain factory functions or a dependency-injection container. Abstract factory earns its keep when *mixing families would break things*, and the type system needs to enforce the grouping.


## Builder

**Intent:** separate the construction of a complex object from its representation, so the same construction process can build different shapes.

**Where it shows up:** SQL query builders, HTTP request builders, configuration objects with twenty optional fields, test fixtures (`UserBuilder().with_email("...").with_role("admin").build()`), DSLs.

**Two reasons to reach for Builder:**

- **Many optional parameters.** A constructor with twelve optional arguments is a usability disaster. A builder lets callers set only what they care about, in any order, with each call self-documenting.
- **Multi-step validation or transformation.** The builder runs validation in `build()`, after every step is set, rather than in the constructor when only some fields are present.

**Python's take:** dataclasses with all-optional fields cover most cases. For fluent chaining, return `self` from each setter. For DSL-style nested construction, callable instances or context managers.

**Kotlin's take:** lambdas with receiver give you a DSL for free. `userBuilder { email = "..."; role = "admin" }` is one of Kotlin's signature idioms.


### Python


In [ ]:
from dataclasses import dataclass, field
from typing import Self


@dataclass
class HttpRequest:
    url: str
    method: str = "GET"
    headers: dict[str, str] = field(default_factory=dict)
    body: str | None = None
    timeout: float = 30.0


# Fluent builder — each setter returns self for chaining.
class HttpRequestBuilder:
    def __init__(self, url: str):
        self._url = url
        self._method = "GET"
        self._headers: dict[str, str] = {}
        self._body: str | None = None
        self._timeout = 30.0

    def method(self, m: str) -> Self:
        self._method = m
        return self

    def header(self, key: str, value: str) -> Self:
        self._headers[key] = value
        return self

    def body(self, b: str) -> Self:
        self._body = b
        return self

    def timeout(self, seconds: float) -> Self:
        self._timeout = seconds
        return self

    def build(self) -> HttpRequest:
        # validation runs once, with the complete state
        if self._method not in {"GET", "POST", "PUT", "DELETE"}:
            raise ValueError(f"unsupported method {self._method}")
        if self._method == "GET" and self._body is not None:
            raise ValueError("GET requests cannot have a body")
        return HttpRequest(
            url=self._url, method=self._method, headers=self._headers,
            body=self._body, timeout=self._timeout,
        )


req = (HttpRequestBuilder("https://api.example.com/users")
       .method("POST")
       .header("Content-Type", "application/json")
       .body('{"name": "Alice"}')
       .timeout(5.0)
       .build())
print(req)


### Kotlin

```kotlin
data class HttpRequest(
    val url: String,
    val method: String = "GET",
    val headers: Map<String, String> = emptyMap(),
    val body: String? = null,
    val timeout: Double = 30.0,
)

// Lambda with receiver — the Kotlin-idiomatic builder.
class HttpRequestBuilder(private val url: String) {
    var method: String = "GET"
    private val headers = mutableMapOf<String, String>()
    var body: String? = null
    var timeout: Double = 30.0

    fun header(key: String, value: String) {
        headers[key] = value
    }

    fun build(): HttpRequest {
        require(method in setOf("GET", "POST", "PUT", "DELETE")) { "unsupported method $method" }
        require(method != "GET" || body == null) { "GET requests cannot have a body" }
        return HttpRequest(url, method, headers.toMap(), body, timeout)
    }
}

fun httpRequest(url: String, block: HttpRequestBuilder.() -> Unit): HttpRequest =
    HttpRequestBuilder(url).apply(block).build()

val req = httpRequest("https://api.example.com/users") {
    method = "POST"
    header("Content-Type", "application/json")
    body = "{"name": "Alice"}"
    timeout = 5.0
}
println(req)
```


**When not to use Builder.** When the constructor is fine — few parameters, no validation, no variation. A `Point(x, y)` does not need a builder. The pattern's overhead is justified when the alternative is a constructor with ten optional arguments, or when validation depends on combinations of fields that can only be checked once construction is complete.


## Prototype

**Intent:** create new objects by cloning an existing instance, rather than constructing from scratch.

**Where it shows up:** game development (clone a fully-configured enemy template, then tweak HP), document templates (clone the default invoice, then fill in customer details), test data (clone a known-good fixture, then mutate one field per test).

**Why clone instead of construct:** sometimes the *configured state* of an existing object is what you want, and reproducing it from raw constructor arguments is either expensive or impractical. Cloning skips the construction sequence entirely.

**Python's take:** the `copy` module gives you `copy()` for shallow clones and `deepcopy()` for recursive clones. No `Cloneable` interface needed.

**Kotlin's take:** `data class .copy()` is built into the language — it generates a copy method that takes named arguments for any fields you want to override. This is one of Kotlin's most beloved features and effectively makes Prototype invisible.

**The shallow-vs-deep distinction matters.** A shallow copy shares nested mutable objects with the original; mutating one mutates both. A deep copy walks the whole object graph and clones everything. Pick the right one for your use case.


### Python


In [ ]:
import copy
from dataclasses import dataclass, field


@dataclass
class EnemyTemplate:
    name: str
    hp: int
    abilities: list[str] = field(default_factory=list)


# Build the template once — expensive setup, registered abilities, etc.
goblin_template = EnemyTemplate(name="Goblin", hp=20, abilities=["bite", "stab"])

# Shallow copy — abilities list is *shared* with the original.
clone_a = copy.copy(goblin_template)
clone_a.hp = 25
clone_a.abilities.append("steal")
print(goblin_template.abilities)  # ['bite', 'stab', 'steal'] — mutated!

# Deep copy — abilities list is independent.
clone_b = copy.deepcopy(goblin_template)
clone_b.abilities.append("hide")
print(goblin_template.abilities)  # ['bite', 'stab', 'steal'] — unchanged by clone_b


# Python-idiomatic alternative: dataclass replace() for immutable updates.
from dataclasses import replace

@dataclass(frozen=True)
class FrozenEnemy:
    name: str
    hp: int
    speed: int = 10


base = FrozenEnemy(name="Goblin", hp=20)
fast = replace(base, speed=20)  # new instance, only `speed` changed
print(base, fast)


### Kotlin

```kotlin
data class Enemy(
    val name: String,
    val hp: Int,
    val abilities: List<String> = emptyList(),
)

// .copy() is generated by the compiler for every data class.
val goblinTemplate = Enemy(name = "Goblin", hp = 20, abilities = listOf("bite", "stab"))

// Clone with one field overridden.
val tougher = goblinTemplate.copy(hp = 40)
println(tougher)   // Enemy(name=Goblin, hp=40, abilities=[bite, stab])

// Override multiple fields, leave the rest the same.
val variant = goblinTemplate.copy(name = "Hobgoblin", hp = 35)
println(variant)

// Because the data class is immutable (val), shared references in nested
// collections are safe — no shallow-vs-deep trap. If you do need to modify
// the abilities, you replace the whole list:
val withMagic = goblinTemplate.copy(abilities = goblinTemplate.abilities + "fireball")
println(withMagic)
```


**When not to use Prototype.** When the constructor is cheap and obvious — just call it. Prototype earns its keep when (a) construction is expensive and the configured state of an existing instance is what you want to start from, or (b) you want a clone-and-modify workflow where most fields stay the same. In Python, prefer `dataclasses.replace()` for the immutable-update case; in Kotlin, `data class .copy()` makes the pattern invisible.


## Choosing between the five

A short decision guide for when more than one pattern fits.

| Question | Pattern |
|---|---|
| "Exactly one instance, accessible everywhere?" | **Singleton** — but try passing the dependency first |
| "Subclasses decide which concrete product to build?" | **Factory Method** |
| "Multiple products that must be from the same family?" | **Abstract Factory** |
| "Many optional fields or multi-step validation?" | **Builder** |
| "Clone a configured instance instead of reconstructing?" | **Prototype** |

Two heuristics worth keeping:

- **The pattern's intent should match your problem, not its structure.** "I have several classes with one method each" looks like Strategy, not Factory. "I want exactly one logger" looks like Singleton, but is often just "I want to inject a logger".
- **Python and Kotlin collapse several of these into language features.** Singleton becomes a module or `object` declaration. Builder becomes a dataclass with defaults or a Kotlin DSL. Prototype becomes `copy.deepcopy()` or `data class .copy()`. The intent vocabulary survives even when the structure has disappeared.


Notebook three turns to **Structural patterns** — the family of seven about *composing* objects into larger structures. Adapter, Bridge, Composite, Decorator, Facade, Flyweight, and Proxy. The recurring theme across all seven is **composition over inheritance** — wiring small objects together to build larger behaviour, rather than extending a class hierarchy.
